# Judge panel smoke run: quality gates G0-G2

Source: `results/smoke_core8/{g0,g1,g2}.json` plus the per-block score cache in
`results/smoke_core8/scores/`, produced on the GPU server by

```
python -u experiments/smoke_core8.py --gate g0
python -u experiments/smoke_core8.py --gate g1
python -u experiments/smoke_core8.py --gate g2
```

This notebook is a **pipeline validation**, not a result about judges. Every
number below answers one question: *are the probabilities we are about to spend
20 model-days producing actually meaningful?* Nothing here is evidence for or
against any claim C1-C8; it is the precondition for collecting that evidence.

The three gates, from the execution plan (sections 46-48):

- **G0** static preparation. Data schema, split leakage, prompt hashes, disk,
  model access. Must pass before any GPU time is spent.
- **G1** one model, 20 items. Are the extracted probabilities finite, normalised,
  deterministic on rerun, and correctly de-swapped?
- **G2** Core-8, 100 stratified items, all seven contexts. Does the response
  tensor build, is retained self-reconstruction zero, is fitting coverage
  monotone, and do the pruners run on it?

Core-20 formal inference starts only after G2 passes.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
SMOKE = os.path.join(ROOT, "results", "smoke_core8")


def load_gate(name):
    path = os.path.join(SMOKE, f"{name}.json")
    if not os.path.exists(path):
        print(f"{name}: not run yet ({path})")
        return None
    with open(path) as handle:
        return json.load(handle)


gates = {name: load_gate(name) for name in ("g0", "g1", "g2")}
for name, payload in gates.items():
    if payload:
        print(f"{name.upper()}: {'PASS' if payload['passed'] else 'FAIL'}")

## Gate results

Only keys prefixed `ok_` are pass/fail conditions; the rest are recorded context.
A gate passes when every one of its `ok_` keys is `True`.

In [ ]:
def checks_frame(payload):
    rows = [
        {"check": k.removeprefix("ok_"), "result": "PASS" if v is True else "FAIL"}
        for k, v in payload["checks"].items() if k.startswith("ok_")
    ]
    return pd.DataFrame(rows).set_index("check")


frames = {n.upper(): checks_frame(p) for n, p in gates.items() if p}
summary = pd.concat(frames, axis=0)
display(summary)

failed = summary[summary["result"] == "FAIL"]
if len(failed):
    print("\nFAILING CHECKS:")
    display(failed)
else:
    print("\nevery registered check passed")

## G0 — what the data actually is

The split manifest is the reproducibility record: it pins which base items are in
FIT / CERT / TEST, and carries a SHA-256 of each split plus the hashes of the
three prompt protocols. If a protocol were reworded after scoring, the hash
comparison here would fail and the already-scored tensor would be known to be
stale rather than silently wrong.

Two dataset facts drive the converter and are worth stating explicitly, because
both are easy to get wrong and neither is documented upstream:

1. RewardBench 2's `id` field **repeats across subsets**, so base items are keyed
   `"{subset}/{id}"`. Grouping on the raw id would merge unrelated tasks and leak
   across splits.
2. The `Ties` subset is **not homogeneous** — `tied:*` rows carry several
   acceptable answers, `ref:*` rows carry one. Tie status is therefore decided by
   the number of chosen responses, never by the subset name.

In [ ]:
g0 = gates["g0"]
c = g0["checks"]

print(f"pairs            {c['n_pairs']}")
print(f"split sizes      {c['split_sizes']}   (base items)")
print(f"contexts         {len(c['contexts'])}")
for name in c["contexts"]:
    print(f"                 {name}")
print(f"\nprotocol hashes  {c['protocol_sha256']}")
print(f"free disk        {c['free_disk_gb']} GB")
print(f"\nprompts to score:")
print(f"  Core-8 smoke   {c['smoke_prompts_core8']:>8,}")
print(f"  Core-20 formal {c['formal_prompts_core20']:>8,}")

## G1 — are the probabilities meaningful?

The response tensor must be reproducible, so labels are **scored, never
generated**: the judge is asked for the sequence log-likelihood of the
continuations `A`, `B` and `C`, and those three numbers are softmaxed. No text is
sampled and no output is parsed, which removes both decoding randomness and
parser failure from the pipeline.

Two failure modes would silently corrupt everything downstream:

- **Tokenization.** `" A"` and `"A"` are different tokens and some tokenizers
  split a label into several pieces. If the three labels do not share a boundary
  convention, the softmax is comparing incomparable quantities. The audit below
  records the exact encoding per model.
- **Orientation.** Under an A/B swap the judge's "A" mass belongs to canonical
  response B. Stored probabilities are always canonical; the involution check
  confirms the remap is applied exactly once.

In [ ]:
g1 = gates["g1"]
c1 = g1["checks"]

print(f"model            {c1['model_id']}   ({c1['n_items']} items)\n")
print("label tokenization:")
display(pd.DataFrame({
    "token": c1["label_tokens"],
    "n_tokens": c1["n_continuation_tokens"],
}))

print(f"max |rerun - first run|   {c1['max_rerun_delta']:.3e}   (0 => bit-identical)")
print(f"mean TV(clean, swapped)   {c1['mean_orientation_tv']:.4f}   (pure position bias)")
print("\naccuracy on the 20-item probe:")
display(pd.Series(c1["accuracy"]).to_frame("value"))

The `mean TV(clean, swapped)` number deserves a note. It is **not** an error: it
is the judge's position bias, measured directly. After canonicalisation both
orderings refer to the same responses, so any remaining difference is the judge
reacting to presentation rather than content. A large value here is exactly the
kind of context-sensitivity the coverage functional is built to preserve — a
judge that is redundant on clean items but distinctive under a swap must be kept.

## G2 — the response tensor

Eight judges, seven contexts, 100 stratified items. The tensor is
`(contexts, items, judges, 3)` and every row is a distribution over
(A better, B better, tie).

Three structural checks matter more than any accuracy number:

- **Self-reconstruction.** A retained judge must reconstruct itself exactly, with
  weight 1 on itself. Anything above ~1e-8 means the minimax LP is not solving
  the problem we think it is.
- **Fitting monotonicity.** Adding a judge to the retained set cannot raise
  fitting coverage. This holds by the simplex embedding and fails loudly if the
  tensor is malformed. (It does *not* hold for held-out error, by design.)
- **Split leakage.** Every pair of a base item lands in one split.

In [ ]:
g2 = gates["g2"]
c2 = g2["checks"]

print(f"tensor shape     {c2['tensor_shape']}  (contexts, items, judges, 3)")
print(f"rows scored      {c2['n_rows']:,} / {c2['n_prompts']:,}   ({c2['success_rate']:.4%})")
print(f"self-recon max   {c2['max_self_reconstruction']:.3e}")
print(f"monotonicity     {c2['monotonicity_violations']} violations")
if c2.get("failures"):
    print("\nfailed blocks:")
    display(pd.DataFrame(c2["failures"]))

### Per-judge behaviour

These are pipeline diagnostics, not a leaderboard. What we are looking for is any
judge whose numbers indicate it is being *scored* wrongly rather than *judging*
badly — a near-uniform `mean_max_probability`, a tie rate of exactly zero, or a
position-bias rate near 1.0 all point at the extraction path, not the model.

In [ ]:
SCORES = os.path.join(SMOKE, "scores")
blocks = []
for fname in sorted(os.listdir(SCORES)):
    if not fname.endswith(".json") or fname.startswith("G1__"):
        continue
    with open(os.path.join(SCORES, fname)) as handle:
        blocks.append(json.load(handle))

acc = pd.DataFrame([
    {"judge": b["judge_id"], "context": b["context"], **b["accuracy"]} for b in blocks
])
three = acc.pivot(index="judge", columns="context", values="three_class_accuracy")
print("three-class accuracy (rows: judge, cols: context)")
display(three.round(3))

per_judge = acc.groupby("judge")[[
    "three_class_accuracy", "binary_accuracy_non_tie",
    "predicted_tie_rate", "gold_tie_rate",
    "mean_max_probability", "position_bias_a_rate",
]].mean().round(3)
print("averaged over contexts")
display(per_judge)

In [ ]:
# MASTER FIGURE.
#   (a) accuracy per judge per context   -- is any judge/context combination broken?
#   (b) confidence vs tie usage          -- degenerate scorers sit in the corners
#   (c) position bias                    -- distance from the 0.5 balanced line
#   (d) pairwise disagreement            -- the quantity coverage actually consumes
judges = sorted({b["judge_id"] for b in blocks})
contexts = [c for c in gates["g0"]["checks"]["contexts"]
            if any(b["context"] == c for b in blocks)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
mat = three.reindex(index=judges, columns=contexts).values
im = ax.imshow(mat, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(contexts)))
ax.set_xticklabels(contexts, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(judges)))
ax.set_yticklabels(judges)
ax.set_title("(a) three-class accuracy")
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center",
                color="w" if mat[i, j] < mat.mean() else "k", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046)

ax = axes[0, 1]
ax.scatter(per_judge["mean_max_probability"], per_judge["predicted_tie_rate"], s=60)
for judge, row in per_judge.iterrows():
    ax.annotate(judge, (row["mean_max_probability"], row["predicted_tie_rate"]),
                fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.axhline(per_judge["gold_tie_rate"].mean(), ls="--", c="grey", lw=1,
           label="gold tie rate")
ax.axvline(1 / 3, ls=":", c="red", lw=1, label="uniform (1/3)")
ax.set_xlabel("mean max probability (confidence)")
ax.set_ylabel("predicted tie rate")
ax.set_title("(b) confidence vs tie usage")
ax.legend(fontsize=8)

ax = axes[1, 0]
bias = per_judge["position_bias_a_rate"].sort_values()
ax.barh(range(len(bias)), bias.values, color="steelblue")
ax.axvline(0.5, ls="--", c="red", lw=1, label="unbiased")
ax.set_yticks(range(len(bias)))
ax.set_yticklabels(bias.index)
ax.set_xlabel("fraction of verdicts = A")
ax.set_title("(c) position bias")
ax.legend(fontsize=8)

ax = axes[1, 1]
by_key = {(b["judge_id"], b["context"]): np.asarray(b["probabilities"]) for b in blocks}
n = len(judges)
disagree = np.zeros((n, n))
for i, a in enumerate(judges):
    for j, b in enumerate(judges):
        vals = [np.abs(by_key[(a, ctx)] - by_key[(b, ctx)]).max(axis=1).mean()
                for ctx in contexts if (a, ctx) in by_key and (b, ctx) in by_key]
        disagree[i, j] = np.mean(vals) if vals else np.nan
im = ax.imshow(disagree, cmap="magma")
ax.set_xticks(range(n)); ax.set_xticklabels(judges, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(judges)
ax.set_title("(d) mean pairwise total variation")
fig.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle("Core-8 smoke run: pipeline diagnostics", fontsize=14)
fig.tight_layout()
plt.show()

Panel (d) is the one that matters for the project. Total variation between two
judges' distributions is exactly what the coverage functional integrates, so this
heatmap is a preview of the redundancy structure: a dark block means those judges
are near-substitutes and the pruner should be able to drop one of them. A panel
that is uniformly bright has no compressible structure and would make the whole
exercise vacuous — which is itself worth knowing before Core-20.

## Pruning on real judges

With eight judges the minimum feasible panel can be found by **exhaustive
enumeration**, so the greedy routes are measured against ground truth rather than
against each other. Any greedy result smaller than the exhaustive optimum would
be a bug, and the assertion at the end checks precisely that.

Note the exact MILP is not used here: `milp_min_representative_set` operates on a
2-D scalar matrix and does not apply to the three-class tensor. At N=8 brute force
is cheap and leaves no doubt.

In [ ]:
rows = []
for gamma, r in c2["pruning"].items():
    for algo in ("backward", "forward", "kswap2"):
        rows.append({
            "gamma": float(gamma), "algorithm": algo,
            "size": len(r[algo]), "optimum": r["exhaustive_size"],
            "gap": len(r[algo]) - r["exhaustive_size"],
            "kept": ", ".join(r[algo]),
        })
prune = pd.DataFrame(rows)
display(prune.pivot(index="algorithm", columns="gamma", values="size"))
print("gap to the exhaustive optimum (0 => provably optimal):")
display(prune.pivot(index="algorithm", columns="gamma", values="gap"))

print("\ncertified optimal panels:")
for gamma, r in c2["pruning"].items():
    print(f"  gamma={float(gamma):<6g} |S*|={r['exhaustive_size']}  "
          f"{', '.join(r['exhaustive_set'])}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
gammas = sorted(prune["gamma"].unique())
opt = [c2["pruning"][str(g)]["exhaustive_size"] for g in gammas]

ax.fill_between(gammas, 0, opt, alpha=0.15, color="grey",
                label="infeasible (below proven optimum)")
for algo, marker in (("backward", "o"), ("forward", "s"), ("kswap2", "^")):
    sub = prune[prune["algorithm"] == algo].sort_values("gamma")
    ax.plot(sub["gamma"], sub["size"], marker=marker, label=algo)
ax.plot(gammas, opt, "k--", lw=2, label="exhaustive optimum")

ax.set_xlabel("tolerance gamma (worst-context TV)")
ax.set_ylabel("retained panel size")
ax.set_title("Core-8: retained size vs tolerance")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

## Verdict

The assertions below are the machine-checkable form of "G2 passed". If this cell
runs clean, the pipeline is cleared for the Core-20 formal run under gate G3
(freeze code commit, model revisions, dataset revision, split manifests and
prompt hashes, then launch).

In [ ]:
for name, payload in gates.items():
    if payload is None:
        continue
    bad = [k for k, v in payload["checks"].items() if k.startswith("ok_") and v is not True]
    assert not bad, f"{name.upper()} failed: {bad}"

assert c2["max_self_reconstruction"] < 1e-8, "retained judges do not reconstruct themselves"
assert c2["monotonicity_violations"] == 0, "fitting coverage is not monotone"
assert c2["success_rate"] >= 0.995, "too many failed rows"
assert (prune["gap"] >= 0).all(), "a greedy pruner beat the exhaustive optimum (bug)"

print("all gates passed -- cleared for the Core-20 formal run (gate G3)")